# PN15 — Full square-root child closure and adult-rung ridge

## TL;DR

The hash-sealed scale-12 target passed both registered arms. The representative square-root children read almost exactly `1.0 / 1.0`, summed to almost `2.0`, and their adult product grew by almost exactly `10×`. The relative-phase curve also transferred, but primes and composites overlap, so it is not a prime classifier.

## Context and methods

Development used scales 8–11. Scale 12 was not calculated until the protocol, source, dependency, validator and development outputs were SHA-256 sealed. The factor coordinate is `x_N(p)=2 log(p)/log(N)`: a factor at `sqrt(N)` reads 1.0, so two square-root children add to 2.0.

In [ ]:
from pathlib import Path
import hashlib
import json
import math

HERE = Path.cwd()
if not (HERE / 'PN15_TARGET_RESULTS.json').exists():
    raise FileNotFoundError('Run this notebook from analysis/primes')

target = json.loads((HERE / 'PN15_TARGET_RESULTS.json').read_text(encoding='utf-8'))
development = json.loads((HERE / 'PN15_DEVELOPMENT_RESULTS.json').read_text(encoding='utf-8'))
template = json.loads((HERE / 'PN15_DEVELOPMENT_TEMPLATE.json').read_text(encoding='utf-8'))
freeze = json.loads((HERE / 'PN15_TARGET_FREEZE_MANIFEST.json').read_text(encoding='utf-8'))
validation = json.loads((HERE / 'PN15_SQRT_ADULT_RIDGE_VALIDATION.json').read_text(encoding='utf-8'))
print(target['test_id'], target['mode'])

## Data integrity

Recompute every pre-target SHA-256 hash before inspecting the target metrics.

In [ ]:
hash_checks = {}
for name, expected in freeze['files'].items():
    observed = hashlib.sha256((HERE / name).read_bytes()).hexdigest()
    hash_checks[name] = observed == expected
assert all(hash_checks.values()), [name for name, ok in hash_checks.items() if not ok]
print(f'{len(hash_checks)} frozen hashes match')

## Adult-rung result

The following cell reconstructs the adjacent adult growths and prints the fresh child ridge.

In [ ]:
periods = {int(row['scale']): float(row['geometry']['median_joint_period']) for row in development['scales']}
periods[12] = float(target['target']['geometry']['median_joint_period'])
growths = {f'{d}->{d+1}': periods[d+1] / periods[d] for d in range(8, 12)}
adult = target['metrics']['full_sqrt_adult_ridge']
summary = {
    'growths': growths,
    'child_A': adult['representative_child_A'],
    'child_B': adult['representative_child_B'],
    'child_ratio': adult['representative_child_A'] / adult['representative_child_B'],
    'child_sum': adult['representative_adult_sum'],
    'adult_fill': adult['target_adult_fill'],
    'verdict': adult['verdict'],
}
print(json.dumps(summary, indent=2))

## Phase transfer and population comparison

The phase template transfers tightly. The prime/composite difference checks whether that curve carries prime-specific information at this grain.

In [ ]:
phase = target['metrics']['phase_transfer']
prime = target['target']['curves']['prime']['means']
composite = target['target']['curves']['composite']['means']
max_difference = max(abs(a - b) for a, b in zip(prime, composite))
phase_summary = {
    'template_correlation': phase['target_template_correlation'],
    'template_rmse': phase['target_template_rmse'],
    'zero_rmse': phase['target_zero_rmse'],
    'wrong_coordinate_rmse': phase['target_wrong_coordinate_template_rmse'],
    'max_prime_composite_sector_difference': max_difference,
    'minimum_primes_per_sector': phase['minimum_target_prime_sector_count'],
}
print(json.dumps(phase_summary, indent=2))

## Results and takeaways

PN15 is a successful registered ARA crosswalk and scale-consistency test. It is not a new prime-location law because selecting children immediately below `sqrt(N)` algebraically drives both coordinates toward 1, their sum toward 2, their product toward N, and adjacent adult growth toward the anchor's 10× step. The phase arm is stable but population-general.

In [ ]:
assert validation['passed'] is True
assert all(validation['checks'].values())
assert adult['verdict'] == 'SUPPORTED'
assert phase['verdict'] == 'SUPPORTED'
print('Independent validator: PASS')

## Recommended next step

Freeze a residual or survivor statistic not guaranteed by square-root selection, then test whether it predicts an untouched prime-survival frequency, location class, or gap class better than raw modular and sieve-informed controls.